# Milestone 1: Data Exploration

This notebook covers:
1. Data download and exploration using DuckDB
2. Corpus preprocessing and parquet export

In [1]:
import sys
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INDICES_DIR = PROJECT_ROOT / "indices"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
INDICES_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY = "All_Beauty"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
META_URL = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"

con = duckdb.connect()

## 1. Data Download & Exploration

Using DuckDB to download and explore the All Beauty metadata.

In [2]:
head_meta = con.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,None
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,None
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,None
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cherioll,[],"{'Brand': 'Cherioll', 'Item Form': 'Powder', '...",B088FKY3VD,None
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Precision,[],{'UPC': '644287689178'},B07NGFDN6G,None


In [3]:
# Count total products (reads from URL)
count_df = con.execute(f"SELECT COUNT(*) AS total FROM read_json_auto('{META_URL}')").df()
print(f"Total products: {count_df['total'][0]:,}")

Total products: 112,590


In [4]:
# Download full metadata and save as parquet locally (skip if already exists)
META_PARQUET = DATA_RAW / "meta_All_Beauty.parquet"
if META_PARQUET.exists():
    print(f"Already exists, skipping download: {META_PARQUET}")
else:
    con.execute(f"""
        COPY (SELECT * FROM read_json_auto('{META_URL}'))
        TO '{META_PARQUET}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print(f"Saved to {META_PARQUET}")

Already exists, skipping download: c:\Users\amato\MDS\b6\575\DSCI_575_project_willchh_jiromig\data\raw\meta_All_Beauty.parquet


In [5]:
# Working from local parquet
meta_df = con.execute(f"""
    SELECT * FROM read_parquet('{DATA_RAW / "meta_All_Beauty.parquet"}')
""").df()

print(f"Shape: {meta_df.shape}")
print(f"\nColumns: {list(meta_df.columns)}")
print("\nNull counts:")
print(meta_df.isnull().sum())

Shape: (112590, 14)

Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

Null counts:
main_category           0
title                   0
average_rating          0
rating_number           0
features                0
description             0
price               94886
images                  0
videos                  0
store               11331
categories              0
details                 0
parent_asin             0
bought_together    112590
dtype: int64


In [6]:
# Inspect a few sample records
for i in range(3):
    print(f"\n--- Product {i+1} ---")
    for key, value in meta_df.iloc[i].items():
        display_val = str(value)[:200] if isinstance(value, (str, list)) else value
        print(f"{key}: {display_val}")


--- Product 1 ---
main_category: All Beauty
title: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)
average_rating: 4.8
rating_number: 10
features: []
description: []
price: nan
images: [{'thumb': 'https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg', 'variant': 'MAIN', 'hi_res': None}
 {'thumb': 'https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'}]
videos: []
store: Howard Products
categories: []
details: {'Package Dimensions': '7.1 x 5.5 x 3 inches; 2.38 Pounds', 'UPC': '617390882781'}
parent_asin: B01CUPMQZE
bought_together: None

--- Product 2 ---
main_category: All Beauty
title: Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.
average_rating: 4.5
rati

## 1b. Reviews Download & Exploration

Downloading product reviews from the same McAuley Lab dataset. Reviews will be aggregated per product and incorporated into the search corpus to enrich retrieval text.

In [7]:
# Preview first 5 reviews (directly from URL)
head_reviews = con.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
print(f"Review columns: {list(head_reviews.columns)}")
head_reviews

Review columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True


In [8]:
# Download reviews and save as parquet (skip if already exists)
REVIEWS_PARQUET = DATA_RAW / "reviews_All_Beauty.parquet"
if REVIEWS_PARQUET.exists():
    print(f"Already exists, skipping download: {REVIEWS_PARQUET}")
else:
    con.execute(f"""
        COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}'))
        TO '{REVIEWS_PARQUET}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print(f"Saved to {REVIEWS_PARQUET}")

# Load and explore
reviews_df = con.execute(f"SELECT * FROM read_parquet('{REVIEWS_PARQUET}')").df()
print(f"\nTotal reviews: {len(reviews_df):,}")
print(f"Shape: {reviews_df.shape}")
print("\nNull counts:")
print(reviews_df.isnull().sum())
print("\nReviews per product:")
print(reviews_df.groupby("parent_asin").size().describe())

Already exists, skipping download: c:\Users\amato\MDS\b6\575\DSCI_575_project_willchh_jiromig\data\raw\reviews_All_Beauty.parquet

Total reviews: 701,528
Shape: (701528, 10)

Null counts:
rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

Reviews per product:
count    112565.000000
mean          6.232204
std          25.189840
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        1962.000000
dtype: float64


## 2. Field Selection & Preprocessing

**Fields selected for retrieval:**
- `title` - primary product identifier
- `description` - detailed product information
- `features` - bullet-point product attributes
- `reviews` - only using the top most helpful review (the review with the most "helpful" votes)

**Rationale:** These fields contain the richest text for matching user queries. Reviews add real user language and practical use cases that product descriptions sometimes miss. Note that price, rating, and images are kept as metadata for display but not included in the search text.

**Preprocessing decisions:**
- Concatenate title, description, features and reviews into a single `text` field per product using `build_text()`
- Skip products with no text content
- For BM25, `tokenize()` applies the following preprocessing pipeline:
  1. Lowercase
  2. Split hyphens and slashes into spaces (e.g. "long-lasting" to "long lasting")
  3. Remove remaining punctuation
  4. Remove English stopwords (NLTK)
  5. Lemmatize each token (NLTK WordNetLemmatizer, e.g. "moisturizers" to "moisturizer")
- For semantic search: use raw concatenated text (the sentence-transformer model handles its own tokenization)

Here are some functions responsible for text preprocessing in the `src/utils` module:

- `tokenize`

```python
def tokenize(text: str) -> list[str]:
    """Lowercase, split hyphens, remove punctuation, remove stopwords, lemmatize."""
    text = text.lower()
    text = re.sub(r"[-/]", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return [LEMMATIZER.lemmatize(t) for t in text.split() if t not in STOPWORDS]
```

- `build_text`

```python
def build_text(product: dict) -> str:
    """Concatenate title + description + features + reviews into a single text field."""
    parts = []
    title = product.get("title")
    if isinstance(title, str) and title.strip():
        parts.append(title)
    parts.extend(_to_str_list(product.get("description")))
    parts.extend(_to_str_list(product.get("features")))
    reviews_text = product.get("reviews_text", "")
    if isinstance(reviews_text, str) and reviews_text.strip():
        parts.append(reviews_text)
    return " ".join(parts)
```


In [ ]:
# Only use the most helpful review
reviews_sorted = reviews_df.sort_values("helpful_vote", ascending=False)
reviews_agg = (
    reviews_sorted
    .groupby("parent_asin")["text"]
    .apply(lambda texts: " ".join(t for t in texts.dropna().head(1) if str(t).strip()))
    .reset_index()
    .rename(columns={"text": "reviews_text"})
)
reviews_lookup = dict(zip(reviews_agg["parent_asin"], reviews_agg["reviews_text"]))
print(f"Products with reviews: {len(reviews_lookup):,}")
print("\nAggregated review text length stats:")
reviews_agg["text_len"] = reviews_agg["reviews_text"].str.len()
print(reviews_agg["text_len"].describe())

Products with reviews: 112,565

Aggregated review text length stats:
count    112565.000000
mean        212.867810
std         313.889224
min           0.000000
25%          53.000000
50%         122.000000
75%         254.000000
max       14989.000000
Name: text_len, dtype: float64


In [10]:
from src.utils import tokenize

# Demo: tokenization pipeline on a sample product text
sample_text = "Long-lasting Moisturizing Shampoo for Dry/Damaged Hair - Sulfate-Free, 16 oz."
print(f"Raw:       {sample_text}")
print(f"Tokenized: {tokenize(sample_text)}")

# Show on a real product from the metadata
real_text = meta_df.iloc[4]["title"]
print(f"\nReal product title: {real_text}")
print(f"Tokenized: {tokenize(real_text)}")

Raw:       Long-lasting Moisturizing Shampoo for Dry/Damaged Hair - Sulfate-Free, 16 oz.
Tokenized: ['long', 'lasting', 'moisturizing', 'shampoo', 'dry', 'damaged', 'hair', 'sulfate', 'free', '16', 'oz']

Real product title: Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers
Tokenized: ['precision', 'plunger', 'bar', 'cartridge', 'grip', '93mm', 'bag', '10', 'plunger']


In [11]:
from src.utils import build_text

records = []
reviews_count = 0
for _, row in meta_df.iterrows():
    product = row.to_dict()
    asin = product.get("parent_asin", "")
    product["reviews_text"] = reviews_lookup.get(asin, "")
    if product["reviews_text"]:
        reviews_count += 1
    text = build_text(product)

    if not text.strip():
        continue
    records.append({
        "parent_asin": asin,
        "title": product.get("title", ""),
        "text": text,
        "price": product.get("price"),
        "average_rating": product.get("average_rating"),
    })

corpus_df = pd.DataFrame(records)
corpus_df.to_parquet(DATA_PROCESSED / "product_corpus.parquet", compression="zstd", index=False)
print(f"Corpus size: {len(corpus_df):,} products")
print(f"Products enriched with reviews: {reviews_count:,}")
print("Saved to data/processed/product_corpus.parquet")
corpus_df.head()

Corpus size: 112,590 products
Products enriched with reviews: 112,417
Saved to data/processed/product_corpus.parquet


,parent_asin,title,text,price,average_rating
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...","Howard LC0008 Leather Conditioner, 8-Ounce (4-...",NaN,4.8
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,NaN,4.5
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),Eye Patch Black Adult with Tie Band (6 Per Pac...,NaN,4.4
3,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...","Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",NaN,3.1
4,B07NGFDN6G,Precision Plunger Bars for Cartridge Grips – 9...,Precision Plunger Bars for Cartridge Grips – 9...,NaN,4.3


In [12]:
# Verify: show a product that was enriched with reviews
enriched = corpus_df[corpus_df["parent_asin"].isin(reviews_lookup.keys())]
print(f"Products with reviews in corpus: {len(enriched):,} / {len(corpus_df):,}")
if len(enriched) > 0:
    sample = enriched.iloc[0]
    print(f"\nExample - {sample['title'][:80]}")
    print(f"Text length: {len(sample['text']):,} chars")
    print(f"\nFull text (first 500 chars):\n{sample['text'][:500]}")

Products with reviews in corpus: 112,565 / 112,590

Example - Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)
Text length: 230 chars

Full text (first 500 chars):
Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack) This is amazing.  I bought one at a local shop....I then looked at Amazon and purchased 5 more.  My beautiful leather sofa and chairs now look like new.  You will not believe it.


In [13]:
print("Price statistics:")
print(corpus_df["price"].describe())
print("\nRating statistics:")
print(corpus_df["average_rating"].describe())

# comparing old text (before review) to new text(with review)
corpus_df["text_len"] = corpus_df["text"].str.len()
corpus_df["base_text_len"] = corpus_df.apply(
    lambda r: len(build_text({"title": r["title"]})), axis=1
)

print("\nText length WITHOUT reviews (title + description + features only):")
print(corpus_df["base_text_len"].describe())
print("\nText length WITH reviews (final corpus):")
print(corpus_df["text_len"].describe())

Price statistics:
count    17704.00000
mean        27.25573
std         50.47202
min          0.01000
25%          9.99000
50%         16.99000
75%         29.90000
max       2548.98000
Name: price, dtype: float64

Rating statistics:
count    112590.000000
mean          3.883488
std           0.874384
min           1.000000
25%           3.400000
50%           4.000000
75%           4.500000
max           5.000000
Name: average_rating, dtype: float64

Text length WITHOUT reviews (title + description + features only):
count    112590.000000
mean        113.579501
std          53.188358
min           0.000000
25%          67.000000
50%         114.000000
75%         158.000000
max        1455.000000
Name: base_text_len, dtype: float64

Text length WITH reviews (final corpus):
count    112590.000000
mean        327.398401
std         319.648444
min           3.000000
25%         166.000000
50%         246.000000
75%         380.000000
max       15024.000000
Name: text_len, dtype: float64


## Final thoughts on EDA

The All Beauty category contains 112,590 products and 701,528 reviews. Price data is sparse (only 15.7% of products have a listed price) but text coverage is wide. Almost every product has a title and 99.9% were enriched with aggregated review text which inceased the median text length from ~114 to ~246 characters (listed under 50%). The reviews per product are heavily right-skewed (median 2 per product and max 1,962) so adding the top most helpful review allows us to get a review (if it exists) despite the number of reviews it recieved. This enriched corpus should give both BM25 and hybrid search more to work with compared to metadata alone.